# Setting-up Oil and Gas Data

In [ ]:
import pandas as pd

# Define Oil Dataframe
oil = pd.read_csv("Data/DCOILWTICO.csv")

oil = oil.rename(columns={
    "observation_date": "Date",
    "DCOILWTICO": "oil_price"
})

oil["Date"] = pd.to_datetime(oil["Date"])
oil["oil_price"] = pd.to_numeric(oil["oil_price"], errors="coerce")

oil = oil.sort_values("Date").set_index("Date")

In [ ]:
# Define Gas Dataframe
gas = pd.read_csv("Data/National_Gas_Price_Weekly.csv")

gas = gas.rename(columns={
    "observation_date": "Date",
    "GASREGW": "gas_price"
})

gas["Date"] = pd.to_datetime(gas["Date"])
gas["gas_price"] = pd.to_numeric(gas["gas_price"], errors="coerce")

gas = gas.sort_values("Date").set_index("Date")

Oil data has missing data points for certain dates. I want to fill them with the average of their neighboring prices

In [3]:
missing_oil = oil[oil["oil_price"].isna()].copy()

print("Missing before fill:")
print(missing_oil.head())
print("Total missing:", len(missing_oil))

Missing before fill:
            oil_price
Date                 
2000-04-21        NaN
2000-05-29        NaN
2000-07-03        NaN
2000-07-04        NaN
2000-09-04        NaN
Total missing: 265


In [4]:
oil["oil_filled"] = oil["oil_price"].interpolate(method="linear")

In [5]:
filled_points = oil.loc[oil["oil_price"].isna(), ["oil_filled"]].copy()
filled_points = filled_points.rename(columns={"oil_filled": "filled_value"})

print("\nFilled values:")
print(filled_points.head())


Filled values:
            filled_value
Date                    
2000-04-21         27.35
2000-05-29         30.06
2000-07-03         31.88
2000-07-04         31.32
2000-09-04         33.67


In [7]:
oil["oil_price"] = oil["oil_filled"]
oil = oil.drop(columns=["oil_filled"])

oil is also measured 3 days earlier than gas, im just going to use the gas dates

In [8]:
oil_weekly = oil.resample("W-MON").mean()

In [ ]:
# This will handle gas starting 1993 but oil starting 2000, 
# will merge only overlapping dates

df_combined = gas.join(oil_weekly, how="inner")

In [11]:
df_combined.head()
df_combined.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1357 entries, 2000-04-10 to 2026-04-06
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   gas_price  1357 non-null   float64
 1   oil_price  1357 non-null   float64
dtypes: float64(2)
memory usage: 31.8 KB


# Granger Causality Check 

In [12]:
df_gc = df_combined.diff().dropna()
df_gc.head()

,gas_price,oil_price
Date,,
2000-04-17,-0.031,0.501333
2000-04-24,-0.007,1.772000
2000-05-01,-0.017,-1.434000
2000-05-08,0.035,1.556000
2000-05-15,0.037,1.888000


Granger test:

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

grangercausalitytests(df_gc[["gas_price", "oil_price"]], maxlag=2)

In [ ]:
grangercausalitytests(df_gc[["oil_price", "gas_price"]], maxlag=2)

## Granger Results Summary

### Oil → Gas

All lags:
p = 0.0000

Strong significance

### Gas → Oil

p-values:

lag 1: 0.016

lag 2: 0.000

lag 3: 0.004

Significant, but weaker

The results show strong evidence that oil prices Granger-cause gasoline prices across all lag lengths, indicating that past oil prices provide significant predictive power for gasoline prices. While gasoline prices also appear to Granger-cause oil prices, this could likely reflects shared market dynamics rather than a true causal relationship. Further analysis is needed.

# Cointegration test

In [17]:
from statsmodels.tsa.stattools import coint

coint_result = coint(df_combined["gas_price"], df_combined["oil_price"])

print("Test stat:", coint_result[0])
print("p-value:", coint_result[1])
print("Critical values:", coint_result[2])

Test stat: -3.6527487090208672
p-value: 0.021050251496756
Critical values: [-3.90453486 -3.34063968 -3.04757921]


p = 0.021 < 0.05 → reject null

-> There IS cointegration

The cointegration test indicates a long-run equilibrium relationship between oil and gasoline prices. Combined with strong Granger causality from oil to gasoline prices, this suggests that a Vector Error Correction Model (VECM) is appropriate to capture both short-run dynamics and long-run equilibrium behavior.